In [1]:
from pathlib import Path
import json
import pandas as pd

def _ensure_cols(df: pd.DataFrame, cols_defaults: dict) -> pd.DataFrame:
    """Ensure df has all columns in cols_defaults; fill missing with defaults."""
    for c, default in cols_defaults.items():
        if c not in df.columns:
            df[c] = default
    return df

def _safe_astype(df: pd.DataFrame, dtype_map: dict) -> pd.DataFrame:
    """Cast only columns that exist; avoid KeyErrors."""
    for c, dt in dtype_map.items():
        if c in df.columns:
            df[c] = df[c].astype(dt)
    return df

def _flatten_threads_to_comments(threads: list) -> pd.DataFrame:
    """
    Convert a list of thread dicts:
      {submission_id, title, ..., comments:[{id, parent_id, body, author, ..., replies:[...]}]}
    into a flat comments DataFrame with consistent schema.
    """
    rows = []

    def walk(node: dict, submission_id: str, depth: int):
        # generic key access
        cid = node.get("comment_id", node.get("id"))
        parent_id = node.get("parent_id", node.get("parent"))
        body = node.get("body")
        author = node.get("author", node.get("user"))

        created = node.get("created_utc")
        score = node.get("score")
        permalink = node.get("permalink")
        subreddit = node.get("subreddit")

        rows.append({
            "comment_id": cid,
            "parent_id": parent_id,
            "submission_id": submission_id,
            "body": body,
            "user": author,
            "created_utc": created,
            "depth": depth,
            "score": score,
            "permalink": permalink,
            "subreddit": subreddit,
        })

        # replies might be under "replies" or "comments" (defensive)
        replies = node.get("replies", [])
        if replies is None:
            replies = []
        for ch in replies:
            walk(ch, submission_id, depth + 1)

    for t in threads:
        sid = t.get("submission_id") or t.get("id")  # defensive
        # thread might store top-level comments under "comments"
        top = t.get("comments", [])
        if top is None:
            top = []
        for c in top:
            walk(c, sid, depth=0)

    return pd.DataFrame(rows)

def load_reddit_generic(path: Path) -> pd.DataFrame:
    """
    Generic loader for:
    - flat comment NDJSON/JSONL (one JSON object per line)
    - thread JSON array (list of threads), including your conversation_threads_*.ndjson
    Returns a flat comments DataFrame with safe, consistent columns.
    """
    path = Path(path)
    if not path.exists():
        return pd.DataFrame(columns=["body", "parent_id", "submission_id", "user", "comment_id", "created_utc", "depth"])

    # Try to detect JSON array vs JSONL by first non-whitespace character
    with path.open("r", encoding="utf-8", errors="replace") as f:
        first_non_ws = ""
        while True:
            ch = f.read(1)
            if not ch:
                break
            if not ch.isspace():
                first_non_ws = ch
                break

    if first_non_ws == "[":
        # JSON array of threads
        with path.open("r", encoding="utf-8") as f:
            threads = json.load(f)
        df = _flatten_threads_to_comments(threads)
    else:
        # JSONL / NDJSON (one object per line)
        df = pd.read_json(path, lines=True)

        # Map common naming variants to your expected schema
        # (do this before casting)
        if "author" in df.columns and "user" not in df.columns:
            df["user"] = df["author"]
        if "id" in df.columns and "comment_id" not in df.columns:
            df["comment_id"] = df["id"]
        if "parent" in df.columns and "parent_id" not in df.columns:
            df["parent_id"] = df["parent"]
        if "link" in df.columns and "submission_id" not in df.columns:
            df["submission_id"] = df["link"]

    # Ensure consistent schema (no KeyErrors later)
    df = _ensure_cols(df, {
        "comment_id": pd.NA,
        "body": pd.NA,
        "submission_id": pd.NA,
        "created_utc": pd.NA,
        "user": pd.NA,
        "parent_id": pd.NA,
        "depth": pd.NA,
    })

    # Safe casting (casts only if present)
    df = _safe_astype(df, {
        "comment_id": "string",
        "body": "string",
        "submission_id": "string",
        "created_utc": "string",
        "user": "string",
        "parent_id": "string",
        "depth": "Int64",   # depth is numeric if we created it; keep nullable int
    }).reset_index(drop=True)

    return df


In [6]:
from pathlib import Path

p_com = Path("data/conversation_threads_beispiel.ndjson")
df_comments = load_reddit_generic(p_com)

print("Loaded comments:", len(df_comments))
print(df_comments.columns.tolist())
df_comments[0:10000]


Loaded comments: 13707
['comment_id', 'parent_id', 'submission_id', 'body', 'user', 'created_utc', 'depth', 'score', 'permalink', 'subreddit']


,comment_id,parent_id,submission_id,body,user,created_utc,depth,score,permalink,subreddit
0,l7ygj6h,1dckxpx,1dckxpx,"I assume that when a law like this is made, it...",Dennis_enzo,1718024663,0,118,/r/changemyview/comments/1dckxpx/cmv_in_the_co...,changemyview
1,l7yh3h3,l7ygj6h,1dckxpx,"Such laws do, of course, include definitions, ...",Grandemestizo,1718024905,1,38,/r/changemyview/comments/1dckxpx/cmv_in_the_co...,changemyview
2,l7yht82,l7yh3h3,1dckxpx,Why isn't that good enough?,Delicious_In_Kitchen,1718025204,2,1,/r/changemyview/comments/1dckxpx/cmv_in_the_co...,changemyview
3,l7yn52s,l7yh3h3,1dckxpx,Not even generally. It's always cosmetic featu...,pcgamernum1234,1718027345,2,18,/r/changemyview/comments/1dckxpx/cmv_in_the_co...,changemyview
4,l7ynq2f,l7yn52s,1dckxpx,I’ve seen laws define it by magazine size,TheGuyThatThisIs,1718027568,3,2,/r/changemyview/comments/1dckxpx/cmv_in_the_co...,changemyview
...,...,...,...,...,...,...,...,...,...,...
9995,lb5baf3,lb5a6s3,1dswvgx,The president isn’t a king. There’s limits to ...,carlse20,1719851145,2,16,/r/changemyview/comments/1dswvgx/cmv_project_2...,changemyview
9996,lb5be6p,lb5a6s3,1dswvgx,To exist in a system of checks and balances. T...,justsomedude717,1719851179,2,18,/r/changemyview/comments/1dswvgx/cmv_project_2...,changemyview
9997,lb5edh3,lb5be6p,1dswvgx,These bureaucrats are all part of the Executiv...,No-Body8448,1719852149,3,1,/r/changemyview/comments/1dswvgx/cmv_project_2...,changemyview
9998,lb5fp06,lb5edh3,1dswvgx,Why do you think that the entire executive bra...,decrpt,1719852572,4,12,/r/changemyview/comments/1dswvgx/cmv_project_2...,changemyview
